In [340]:
import numpy as np
import pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.feature_selection import mutual_info_regression
from sklearn.feature_selection import SelectKBest,VarianceThreshold,RFECV
from sklearn.linear_model import LassoCV
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline



In [341]:
df=pd.read_csv("D:\\price_revalue_stystem\\data\\03-features\\train_df.csv")

df.head()


,Year,Distance,Owner,Fuel,Drive,Type,brand,model,State,age_of_car,...,Type_HatchBack,Type_Lux_SUV,Type_Lux_sedan,Type_SUV,Type_Sedan,Fuel_Cng,Fuel_Diesel,Fuel_Lpg,Fuel_Petrol,Price_log
0,2017.0,58647,1,Petrol,Manual,HatchBack,MARUTI,ALTO,Telangana,9.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,12.721889
1,2015.0,79288,1,Petrol,Automatic,Sedan,VOLKSWAGEN,VENTO,Maharashtra,11.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,13.124363
2,2022.0,7396,1,Petrol,Manual,HatchBack,MARUTI,SWIFT,Tamil Nadu,4.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,13.543703
3,2020.0,22814,2,Petrol,Manual,SUV,HYUNDAI,VENUE,Karnataka,6.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,13.739710
4,2017.0,28696,1,Petrol,Manual,HatchBack,HYUNDAI,ELITE,Maharashtra,9.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,13.204866


In [342]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6240 entries, 0 to 6239
Data columns (total 27 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Year             6240 non-null   float64
 1   Distance         6240 non-null   int64  
 2   Owner            6240 non-null   int64  
 3   Fuel             6240 non-null   object 
 4   Drive            6240 non-null   object 
 5   Type             6240 non-null   object 
 6   brand            6240 non-null   object 
 7   model            6240 non-null   object 
 8   State            6240 non-null   object 
 9   age_of_car       6240 non-null   float64
 10  km_per_year      6240 non-null   float64
 11  age_x_distance   6240 non-null   float64
 12  brand_count      6240 non-null   int64  
 13  model_count      6240 non-null   int64  
 14  is_luxury        6240 non-null   int64  
 15  Drive_Automatic  6240 non-null   float64
 16  Drive_Manual     6240 non-null   float64
 17  Type_HatchBack

# Relation ship

# chi_square

In [343]:

from scipy.stats import chi2_contingency
from itertools import combinations

def chi_square_relationships(df, columns, alpha=0.05):
    results = []

    for col1, col2 in combinations(columns, 2):
        # Create contingency table
        table = pd.crosstab(df[col1], df[col2])
        
        # Perform chi-square test
        chi2, p, dof, expected = chi2_contingency(table)
        
        results.append({
            "Variable 1": col1,
            "Variable 2": col2,
            "Chi-square": chi2,
            "p-value": p,
            "Degrees of Freedom": dof,
            "Significant": "Yes" if p < alpha else "No"
        })

    return pd.DataFrame(results)


In [344]:
columns = ['Drive', 'brand', 'Type', 'Fuel', 'State']

result_df = chi_square_relationships(df, columns)

result_df


,Variable 1,Variable 2,Chi-square,p-value,Degrees of Freedom,Significant
0,Drive,brand,160.001190,2.457260e-26,15,Yes
1,Drive,Type,237.693352,2.911860e-50,4,Yes
2,Drive,Fuel,58.037441,1.543201e-12,3,Yes
3,Drive,State,135.089033,6.564550e-21,16,Yes
4,brand,Type,6153.363364,0.000000e+00,60,Yes
5,brand,Fuel,734.795503,1.740371e-125,45,Yes
6,brand,State,608.024042,7.966247e-34,240,Yes
7,Type,Fuel,1171.942650,1.904595e-243,12,Yes
8,Type,State,150.686135,5.972003e-09,64,Yes
9,Fuel,State,494.650147,1.832113e-75,48,Yes


# anova

In [345]:

from scipy.stats import f_oneway

def anova_test(df, categorical_col, numeric_col, alpha=0.05):
    
    # Drop missing values
    df_clean = df[[categorical_col, numeric_col]].dropna()
    
    # Create groups
    groups = [
        group[numeric_col].values 
        for name, group in df_clean.groupby(categorical_col)
    ]
    
    # Perform ANOVA
    f_stat, p_value = f_oneway(*groups)
    
    result = pd.DataFrame({
        "Categorical Variable": [categorical_col],
        "Numeric Variable": [numeric_col],
        "F-Statistic": [f_stat],
        "p-value": [p_value],
        "Significant": ["Yes" if p_value < alpha else "No"]
    })
    
    return result


In [346]:
result = anova_test(df, 'Fuel', 'Price_log')
print(result)


  Categorical Variable Numeric Variable  F-Statistic        p-value  \
0                 Fuel        Price_log   178.061936  8.982733e-111   

  Significant  
0         Yes  


In [347]:
result = anova_test(df, 'Drive', 'Price_log')
print(result)

  Categorical Variable Numeric Variable  F-Statistic        p-value  \
0                Drive        Price_log   882.109911  1.985597e-181   

  Significant  
0         Yes  


In [348]:
result = anova_test(df, 'Type', 'Price_log')
print(result)

  Categorical Variable Numeric Variable  F-Statistic  p-value Significant
0                 Type        Price_log   790.769287      0.0         Yes


In [349]:
result = anova_test(df, 'brand', 'Price_log')
print(result)

  Categorical Variable Numeric Variable  F-Statistic        p-value  \
0                brand        Price_log    82.668131  6.705189e-232   

  Significant  
0         Yes  


In [350]:
result = anova_test(df, 'model', 'Price_log')
print(result)

  Categorical Variable Numeric Variable  F-Statistic  p-value Significant
0                model        Price_log   120.476738      0.0         Yes


In [351]:
result = anova_test(df, 'State', 'Price_log')
print(result)

  Categorical Variable Numeric Variable  F-Statistic        p-value  \
0                State        Price_log    33.666233  7.797960e-100   

  Significant  
0         Yes  


# Variance threshold

In [352]:
from sklearn.feature_selection import VarianceThreshold

def variance_threshold_selector(df, threshold=0.01):
    
    X = df
    
    selector = VarianceThreshold(threshold=threshold)
    X_selected = selector.fit_transform(X)
    
    selected_cols = X.columns[selector.get_support()]
    removed_cols = X.columns[~selector.get_support()]
    
    return selected_cols.tolist(), removed_cols.tolist()


In [353]:
data=df[["Distance","km_per_year","age_x_distance","brand_count","model_count"]]
selected, removed = variance_threshold_selector(data, threshold=0.01)

print("Selected:", selected)
print("Removed:", removed)


Selected: ['Distance', 'km_per_year', 'age_x_distance', 'brand_count', 'model_count']
Removed: []


# Mutual_info()

In [354]:
train_df=pd.read_csv("D:\\price_revalue_stystem\\data\\03-features\\train_df.csv")
test_df=pd.read_csv("D:\\price_revalue_stystem\\data\\03-features\\test_df.csv")
df=pd.concat([train_df, test_df], ignore_index=True)
df=df.drop(columns=['Fuel', 'Drive', 'Type', 'brand', 'model','State'], axis=1)


In [355]:

from sklearn.feature_selection import mutual_info_regression

def mutual_info_feature_selection(df, target_col):
 
    x=df.drop(target_col, axis=1)
    y = df[target_col]
    mi_scores = mutual_info_regression(x, y, random_state=42)
    
    mi_df = pd.DataFrame({
        'Feature': x.columns,
        'MI Score': mi_scores
    }).sort_values(by='MI Score', ascending=False)
    
    return mi_df
mi_results = mutual_info_feature_selection(df, 'Price_log')
print(mi_results)


            Feature  MI Score
7       model_count  0.387483
3        age_of_car  0.242240
0              Year  0.241702
11   Type_HatchBack  0.182885
14         Type_SUV  0.118059
5    age_x_distance  0.100889
6       brand_count  0.089861
10     Drive_Manual  0.069133
9   Drive_Automatic  0.068196
17      Fuel_Diesel  0.042551
4       km_per_year  0.034800
15       Type_Sedan  0.033375
19      Fuel_Petrol  0.028698
12     Type_Lux_SUV  0.020575
1          Distance  0.017174
2             Owner  0.012162
16         Fuel_Cng  0.011938
13   Type_Lux_sedan  0.006650
8         is_luxury  0.004246
18         Fuel_Lpg  0.000368


In [356]:
# df.groupby('Fuel')['Price_log'].mean()
# import pingouin as pg
# pg.anova(dv='Price_log', between='Fuel', data=df)


# Lasso

In [357]:

from sklearn.linear_model import LassoCV
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline


In [358]:

X = df.drop('Price_log', axis=1)
y = df['Price_log']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [359]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('lasso', LassoCV(cv=5, random_state=42))
])

pipeline.fit(X_train, y_train)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('scaler', ...), ('lasso', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"eps eps: float, default=1e-3Length of the path. ``eps=1e-3`` means that``alpha_min / alpha_max = 1e-3``.",0.001
,"n_alphas n_alphas: int, default=100Number of alphas along the regularization path... deprecated:: 1.7 `n_alphas` was deprecated in 1.7 and will be removed in 1.9. Use `alphas` instead.",'deprecated'
,"alphas alphas: array-like or int, default=NoneValues of alphas to test along the regularization path.If int, `alphas` values are generated automatically.If array-like, list of alpha values to use... versionchanged:: 1.7 `alphas` accepts an integer value which removes the need to pass `n_alphas`... deprecated:: 1.7 `alphas=None` was deprecated in 1.7 and will be removed in 1.9, at which point the default value will be set to 100.",'warn'
,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto false, no intercept will be used in calculations(i.e. data is expected to be centered).",True


In [360]:
lasso = pipeline.named_steps['lasso']

coefficients = pd.Series(
    lasso.coef_,
    index=X.columns
)

selected_features = coefficients[coefficients != 0]

print("Selected Features:")
print(selected_features.sort_values(ascending=False))


Selected Features:
Distance           1.641156e-01
Year               1.372026e-01
Drive_Automatic    8.995370e-02
model_count        6.331950e-02
Type_SUV           5.237843e-02
Fuel_Diesel        4.873440e-02
Type_Lux_SUV       4.621225e-02
Type_Lux_sedan     2.607404e-02
is_luxury          7.824227e-03
Fuel_Lpg           4.668906e-04
Drive_Manual      -5.951080e-15
Fuel_Petrol       -6.623233e-03
Owner             -2.475669e-02
brand_count       -3.660611e-02
age_of_car        -6.548577e-02
km_per_year       -1.088142e-01
age_x_distance    -1.195180e-01
Type_HatchBack    -1.458675e-01
dtype: float64


In [361]:
from sklearn.metrics import mean_squared_error

pred = pipeline.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, pred))

print("RMSE:", rmse)


RMSE: 0.26160297998782245


In [362]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)

rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
print("Random Forest RMSE:", rf_rmse)


Random Forest RMSE: 0.18689504547856728


In [369]:
rf_importance = pd.Series(
    rf.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

imp=pd.DataFrame({
    'Feature': rf_importance.index,
    'Importance': rf_importance.values
}).sort_values(by='Importance', ascending=False).head(15)
imp

,Feature,Importance
0,Type_HatchBack,0.274338
1,age_of_car,0.149712
2,model_count,0.142689
3,Year,0.117978
4,age_x_distance,0.065762
5,brand_count,0.051840
6,Type_Sedan,0.042423
7,km_per_year,0.036466
8,Distance,0.035291
9,Drive_Manual,0.026580


In [370]:
print(imp["Feature"].head(15))

0      Type_HatchBack
1          age_of_car
2         model_count
3                Year
4      age_x_distance
5         brand_count
6          Type_Sedan
7         km_per_year
8            Distance
9        Drive_Manual
10    Drive_Automatic
11              Owner
12        Fuel_Diesel
13           Type_SUV
14        Fuel_Petrol
Name: Feature, dtype: object


In [371]:
from xgboost import XGBRegressor

xgb = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

xgb.fit(X_train, y_train)

xgb_pred = xgb.predict(X_test)

xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_pred))
print("XGBoost RMSE:", xgb_rmse)


XGBoost RMSE: 0.17582704691629453


In [372]:
print("RF % Error:", (np.exp(rf_rmse) - 1) * 100)
print("XGB % Error:", (np.exp(xgb_rmse) - 1) * 100)


RF % Error: 20.550075565522818
XGB % Error: 19.223184064483224


In [375]:
df.head()

,Year,Distance,Owner,age_of_car,km_per_year,age_x_distance,brand_count,model_count,is_luxury,Drive_Automatic,...,Type_HatchBack,Type_Lux_SUV,Type_Lux_sedan,Type_SUV,Type_Sedan,Fuel_Cng,Fuel_Diesel,Fuel_Lpg,Fuel_Petrol,Price_log
0,2017.0,58647,1,9.0,5864.700000,527823.0,2689,416,0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,12.721889
1,2015.0,79288,1,11.0,6607.333333,872168.0,131,40,0,1.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,13.124363
2,2022.0,7396,1,4.0,1479.200000,29584.0,2689,620,0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,13.543703
3,2020.0,22814,2,6.0,3259.142857,136884.0,1559,57,0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,13.739710
4,2017.0,28696,1,9.0,2869.600000,258264.0,1559,301,0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,13.204866


In [377]:
df.to_csv("D:\\price_revalue_stystem\\data\\04-predictionsfeature_selected.csv", index=False)